# WRDS IBES Pull — TSLA Analyst Data

Pulls historical analyst data for TSLA from WRDS IBES (Refinitiv's analyst forecast database). Works on any Linux machine with internet, no desktop client required.

**Prerequisites:**
- WRDS account through GMU (request at https://wrds-www.wharton.upenn.edu/ if you don't have one).
- IBES subscription (most universities have it).

**Output:** `wrds_tsla_analyst.csv` — one row per `statpers` date, with consensus EPS, price target, and recommendation fields merged.

## 1. Install and connect

In [ ]:
!pip install -q wrds pandas

In [ ]:
import wrds
import pandas as pd

# First time: this prompts for your WRDS username + password
# and offers to save a .pgpass file so future runs are passwordless.
db = wrds.Connection()

print("Connected. Available libraries you can query:")
libs = db.list_libraries()
print([l for l in libs if 'ibes' in l.lower()])

## 2. Confirm IBES tables are accessible

In [ ]:
tables = db.list_tables(library='ibes')
print("IBES tables (showing relevant ones):")
for t in sorted(tables):
    if any(k in t for k in ('statsum', 'recd', 'ptg')):
        print(" ", t)

## 3. Configuration

In [ ]:
TICKER     = 'TSLA'
START_DATE = '2025-07-01'
END_DATE   = '2026-04-28'
OUTPUT_CSV = 'wrds_tsla_analyst.csv'

## 4. Pull EPS forecast summary (next quarter)

`statsum_epsus` is the consensus statistics history. `fpi='1'` = next fiscal quarter forecast. Each row is a snapshot on `statpers` (third Thursday of each month, the IBES summary date).

In [ ]:
eps_q = db.raw_sql(f"""
    SELECT ticker, statpers, fpedats, fpi,
           meanest, medest, stdev, numest, highest, lowest
    FROM ibes.statsum_epsus
    WHERE ticker = '{TICKER}'
      AND fpi = '1'
      AND statpers BETWEEN '{START_DATE}' AND '{END_DATE}'
    ORDER BY statpers
""")

eps_q = eps_q.rename(columns={
    'meanest':  'eps_q_mean',
    'medest':   'eps_q_median',
    'stdev':    'eps_q_stdev',
    'numest':   'eps_q_n',
    'highest':  'eps_q_high',
    'lowest':   'eps_q_low',
    'fpedats':  'eps_q_period_end',
})
eps_q = eps_q.drop(columns=['fpi'])
print(eps_q.shape)
eps_q.head()

## 5. Pull EPS forecast summary (current fiscal year, FY1)

In [ ]:
eps_fy = db.raw_sql(f"""
    SELECT ticker, statpers, fpedats,
           meanest AS eps_fy1_mean,
           medest  AS eps_fy1_median,
           stdev   AS eps_fy1_stdev,
           numest  AS eps_fy1_n
    FROM ibes.statsum_epsus
    WHERE ticker = '{TICKER}'
      AND fpi = '6'
      AND statpers BETWEEN '{START_DATE}' AND '{END_DATE}'
    ORDER BY statpers
""")
eps_fy = eps_fy.drop(columns=['fpedats'])
print(eps_fy.shape)
eps_fy.head()

## 6. Pull recommendation summary (Buy/Hold/Sell consensus)

In [ ]:
recs = db.raw_sql(f"""
    SELECT ticker, statpers,
           meanrec   AS rec_mean,
           numrec    AS rec_n,
           buypercent AS pct_buy,
           holdpercent AS pct_hold,
           sellpercent AS pct_sell
    FROM ibes.recdsum
    WHERE ticker = '{TICKER}'
      AND statpers BETWEEN '{START_DATE}' AND '{END_DATE}'
    ORDER BY statpers
""")
print(recs.shape)
recs.head()

## 7. Pull price target summary

In [ ]:
pts = db.raw_sql(f"""
    SELECT ticker, statpers,
           meanptg AS pt_mean,
           medptg  AS pt_median,
           hightg  AS pt_high,
           lowtg   AS pt_low,
           numest  AS pt_n
    FROM ibes.ptgsum
    WHERE ticker = '{TICKER}'
      AND statpers BETWEEN '{START_DATE}' AND '{END_DATE}'
    ORDER BY statpers
""")
print(pts.shape)
pts.head()

## 8. Merge everything on (ticker, statpers)

In [ ]:
merged = eps_q.merge(eps_fy, on=['ticker', 'statpers'], how='outer')\
              .merge(recs,    on=['ticker', 'statpers'], how='outer')\
              .merge(pts,     on=['ticker', 'statpers'], how='outer')\
              .sort_values('statpers')\
              .reset_index(drop=True)

print('Final shape:', merged.shape)
print('Date range :', merged['statpers'].min(), '->', merged['statpers'].max())
merged.head()

In [ ]:
merged.to_csv(OUTPUT_CSV, index=False)
print(f'Saved {len(merged)} rows to {OUTPUT_CSV}')

print('\nMissing values per column:')
print(merged.isna().sum())

In [ ]:
db.close()

## 9. Notes

- **statpers** is the IBES summary snapshot date (third Thursday of each month). For backtesting, I will forward-fill these monthly snapshots to daily so each trading day uses the most recent consensus available *as of that day*. No lookahead.
- If `recdsum` columns differ in your version of WRDS (some installs use `buypct` instead of `buypercent`), run `db.describe_table(library='ibes', table='recdsum')` to inspect the actual column names and adjust the SQL.
- If `statsum_epsus` returns nothing for `fpi='1'`, try `fpi=1` (integer) instead — schema varies.
- Send me `wrds_tsla_analyst.csv` and I will integrate it into the backtest.